# Systematic Trading Strategies with ML — Meta-Model Submission

**Imperial College London × Alken Asset Management.** A *meta-model* that takes the provided primary
trading signal `s ∈ {-1, 0, +1}` for 11 futures and predicts, for each non-zero signal, the
**probability that following the trade is profitable** under a triple-barrier exit.

This notebook is the full deliverable. **Part I (this file so far)** establishes the *foundations* the
model consumes, in four steps:

1. **Exploratory data analysis** — the raw price panel and the character of the primary signal.
2. **External macro dataset** — the Bloomberg cross-asset series we added (feature family **F11**).
3. **HMM regime features** — the learned volatility-regime states (families **F3** and **F17**).
4. **Feature engineering** — the full catalogue of feature families (**F1–F17**) and how they are built.

The later parts — triple-barrier *labelling*, *meta-model fitting*, and *position-sizing / weight
extraction* — follow these foundations.

> **Scope / non-duplication.** This notebook owns the **raw-data and signal EDA** plus the **feature
> foundations**. The downstream *modelling* diagnostics (feature-redundancy clustering, per-family
> missingness, cluster-level importance) live in the model-development notebook `metamodel.ipynb`
> (§1/§7) and are not repeated here.

## Section 0 — Setup & data load

In [ ]:
%matplotlib inline
import json, re, collections, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
np.random.seed(42); SEED = 42
plt.rcParams["figure.dpi"] = 100
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 40)

from stml.io import _find_repo_root, load_clean_data, load_returns_panel
from stml import na_checks as nc
from stml.model.dataset import load_matrix, close_panel, events_frame, asset_class_map

ROOT = _find_repo_root(Path.cwd().resolve())
DATA, RESULTS, REPORTS = ROOT / "data", ROOT / "results", ROOT / "reports"

ohlcv, signals = load_clean_data()      # ohlcv: long OHLCV ; signals: wide (date + 11 instrument cols)
matrix  = load_matrix()                 # the engineered feature matrix (nonzero-signal trade-days)
close_w = close_panel()                 # wide close prices, date-indexed, instrument columns
ret_w   = load_returns_panel()          # wide daily LOG returns, full price history
sig_w   = signals.set_index("date")     # wide signals, date-indexed
INSTR   = list(nc.INSTRUMENTS)          # canonical 11-instrument order
CLASS   = asset_class_map()             # {instrument -> 'EQ' / 'EN' / 'ME'}
prov    = json.load(open(RESULTS / "feature_matrix_provenance.json"))

TICKER_NAMES = {
    "cl1s": "Crude Oil (CL)",   "es1s": "S&P 500 e-mini (ES)", "fesx1s": "Euro Stoxx 50 (FESX)",
    "gc1s": "Gold (GC)",        "hg1s": "Copper (HG)",         "ho1s": "Heating Oil (HO)",
    "ng1s": "Natural Gas (NG)", "nq1s": "Nasdaq 100 (NQ)",     "pl1s": "Platinum (PL)",
    "rb1s": "RBOB Gasoline (RB)", "si1s": "Silver (SI)",
}

assert prov["fe_train_end_date"] == "2021-07-01"
print(f"matrix: {matrix.shape[0]:,} nonzero-signal rows x {matrix.shape[1]} cols "
      f"({prov['n_feature_cols']} features + 4 metadata)")
print(f"instruments: {len(INSTR)} | FE-train boundary <= {prov['fe_train_end_date']} | seed {prov['seed']}")
print(f"signal window: {sig_w.index.min().date()} -> {sig_w.index.max().date()}")

In [ ]:
# Capability probe — the HMM transition-matrix display (Section 3) needs a live fit, which requires
# `hmmlearn`. Everything else (incl. the already-materialised f3_*/f17_* columns) runs without it.
try:
    import hmmlearn  # noqa: F401
    CAP_HMM = True
except Exception:
    CAP_HMM = False
print("hmmlearn available:", CAP_HMM, "" if CAP_HMM else "(run `uv sync --group features-extra`)")

## Section 1 — Exploratory data analysis

### 1.1 — Coverage: a two-window dataset

The 11 instruments span three asset classes — **Equity** (`es1s`, `nq1s`, `fesx1s`), **Energy**
(`cl1s`, `ho1s`, `rb1s`, `ng1s`) and **Metals** (`gc1s`, `si1s`, `hg1s`, `pl1s`). The commodity
contracts carry ~30 years of price history; the equity indices start later. Critically, the **primary
signal** only exists for **2020-01 → 2022-06**. The decades of price history exist *only* to warm up
trailing features causally — all modelling happens inside the short signal window.

In [ ]:
# Data-availability heatmap (one row per instrument; red lines mark the signal window).
avail = close_w.notna().astype(int).T          # instrument x date
fig, ax = plt.subplots(figsize=(13, 3.2))
ax.imshow(avail.values, aspect="auto", cmap="Greys", interpolation="nearest")
ax.set_yticks(range(len(avail.index))); ax.set_yticklabels(avail.index)
yrs = pd.to_datetime(avail.columns).year
yr_changes = np.where(np.diff(yrs) != 0)[0] + 1
step = max(1, len(yr_changes) // 12)
ax.set_xticks(yr_changes[::step]); ax.set_xticklabels([str(yrs[i]) for i in yr_changes[::step]], rotation=45)
s0 = avail.columns.get_indexer([signals.date.min()], method="nearest")[0]
s1 = avail.columns.get_indexer([signals.date.max()], method="nearest")[0]
ax.axvline(s0, color="C3", lw=1.2, label="signal window"); ax.axvline(s1, color="C3", lw=1.2)
ax.set_title("OHLCV data availability per instrument  (red = signal window)")
ax.legend(loc="lower left"); plt.tight_layout(); plt.show()

n_eq = sum(CLASS[i] == "EQ" for i in INSTR)
print(f"{len(INSTR)} instruments: {len(INSTR)-n_eq} commodities (~30y history), {n_eq} equity indices; "
      f"signal window {sig_w.index.min().date()} -> {sig_w.index.max().date()}")

### 1.2 — Return structure: stationary but heavy-tailed

Daily log returns are **stationary** (all instruments reject the ADF unit-root null) but markedly
**non-Gaussian**: excess kurtosis is well above 0 everywhere (fat tails). This motivates robust /
tree-based models downstream and the **volatility-scaled** triple-barrier exits used for labelling.

In [ ]:
# Per-instrument return summary over the full price history (log returns).
ANN = np.sqrt(252)
ret_summary = pd.DataFrame({
    "name":          pd.Series(TICKER_NAMES),
    "mean_daily_bp": ret_w.mean() * 1e4,
    "ann_vol_%":     ret_w.std() * ANN * 100,
    "skew":          ret_w.skew(),
    "kurtosis":      ret_w.kurt(),          # excess kurtosis; >> 0 = fat tails
    "min_day_%":     ret_w.min() * 100,
    "max_day_%":     ret_w.max() * 100,
    "n_obs":         ret_w.count(),
}).reindex(INSTR).round(2)
display(ret_summary)

In [ ]:
# Log-return distributions with a Gaussian overlay (log y-axis exposes the tails).
fig, axes = plt.subplots(3, 4, figsize=(14, 8))
for ax, inst in zip(axes.flat, INSTR):
    r = ret_w[inst].dropna()
    ax.hist(r, bins=80, density=True, alpha=0.7, color="C0")
    xs = np.linspace(r.quantile(0.001), r.quantile(0.999), 200)
    ax.plot(xs, (1/(r.std()*np.sqrt(2*np.pi))) * np.exp(-0.5*((xs-r.mean())/r.std())**2),
            color="C3", lw=1.2, label="N(mu,sig^2)")
    ax.set_title(f"{inst}  kurt={r.kurt():.1f}", fontsize=10); ax.set_yscale("log")
    ax.legend(fontsize=7)
for ax in axes.flat[len(INSTR):]:
    ax.set_visible(False)
fig.suptitle("Log-return distributions (log y, Gaussian overlay) — fat tails everywhere", y=1.02)
plt.tight_layout(); plt.show()

### 1.3 — Cross-sectional structure: three blocks

Clustering the full-sample log-return correlation matrix surfaces **three blocks** — energy, metals
and equity — with intra-block correlation ~0.7–0.9 and weak, regime-dependent cross-block structure.
This panel structure is why cross-validation must be **blocked** (a fold leaking one block leaks its
neighbours) and why the cross-sectional **F9** features are computed over the whole universe.

In [ ]:
import seaborn as sns
# corr_max_info: pairwise-complete (max data, not truncated to shortest history) + PSD-repaired.
corr = nc.corr_max_info(ret_w, min_periods=252)
g = sns.clustermap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1,
                   figsize=(8, 8), cbar_pos=(0.02, 0.85, 0.03, 0.12))
g.fig.suptitle("Daily log-return correlation (pairwise-complete, PSD-repaired, clustered)", y=1.02)
plt.show()

### 1.4 — The primary signal: imbalanced, persistent, near-independent

The signal is highly **imbalanced and instrument-specific** (`ho1s` is flat ~90% of days, `ng1s` is
**never long**, `es1s` is long ~70%), and highly **persistent** (lag-1 autocorrelation 0.6–0.9, mean
run length > 5 days). Persistence means the rows are *not* i.i.d. — **purged / blocked CV is
mandatory**. Cross-instrument signal correlation is low, so each instrument carries near-independent
information.

In [ ]:
# Class balance + persistence per instrument.
long_sig = signals.melt(id_vars="date", var_name="instrument", value_name="s")
bal = (long_sig.groupby("instrument")["s"].value_counts(normalize=True)
       .unstack(fill_value=0).round(3))
bal.columns = [f"p(s={int(c)})" for c in bal.columns]
rows = []
for inst in INSTR:
    s = sig_w[inst]
    runs = (s != s.shift()).cumsum()
    rl = s[s != 0].groupby(runs).size()
    rows.append({"instrument": inst, "lag1_autocorr": round(s.autocorr(1), 3),
                 "mean_run_len": round(rl.mean(), 1) if len(rl) else 0.0,
                 "nonzero_frac": round((s != 0).mean(), 3)})
persist = pd.DataFrame(rows).set_index("instrument")
display(bal.join(persist).reindex(INSTR))

# Stacked class-share bar.
shares = pd.DataFrame({
    "short (-1)": (sig_w == -1).mean(), "flat (0)": (sig_w == 0).mean(), "long (+1)": (sig_w == 1).mean(),
}).reindex(INSTR)
ax = shares.plot(kind="bar", stacked=True, figsize=(11, 3.6),
                 color=["#d62728", "#bdbdbd", "#2ca02c"])
ax.set_ylabel("share of days"); ax.set_ylim(0, 1)
ax.set_title("Signal class balance per instrument (2020–2022)")
ax.legend(loc="center left", bbox_to_anchor=(1.0, 0.5)); plt.tight_layout(); plt.show()

### 1.5 — Signal × forward returns: the signal predicts the *next* bar

The single most important EDA finding for labelling. Two complementary views agree:

- **Lead/lag** `corr(s_t, r_{t+k})` is strongest at **k = +1** — the signal *leads* the return.
- The **raw signed-return Sharpe by lag** (`g_i(L) = s_i · u_{t+L}`, from the labelling study) is
  *negative* at **lag 0** (contemporaneous) for most instruments, **strongest at lag 1**, and decays by
  lag 2 — a textbook **placebo-in-time** pattern.

Together: the signal is a **short-horizon mean-reversion / counter-trend** call that pays off on the
**next** bar. The load-bearing labelling consequence is to **enter at `t+1`, not `t`** (entering at `t`
would be look-ahead). The three thin / awkward names to watch are **`cl1s`** (strongest, label-1≈0.70),
**`ng1s`** (short-only) and **`ho1s`** (~60 events, statistically weak).

In [ ]:
# (a) Per-instrument corr(s_t, r_{t+1}), mean pnl and hit rate.
rets_s = close_w.pct_change()
hit = []
for inst in INSTR:
    s = sig_w[inst].reindex(close_w.index); fwd = rets_s[inst].shift(-1)
    m = (s != 0) & fwd.notna(); pnl = (s * fwd)[m]
    hit.append({"instrument": inst, "corr_s_rfwd": round(float(s[m].corr(fwd[m])), 3),
                "mean_pnl_bp": round(float(pnl.mean()) * 1e4, 2),
                "hit_rate": round(float((pnl > 0).mean()), 3), "n": int(m.sum())})
display(pd.DataFrame(hit).set_index("instrument").reindex(INSTR))
print("corr(s_t, r_{t+1}) is mostly positive -> the signal predicts the NEXT bar (counter-trend flavour).")

In [ ]:
# (b) Lead/lag: corr(signal_t, r_{t+k}) for k in [-5..5].  k>0 = signal leads return = predictive.
lags = range(-5, 6)
lead_lag = pd.DataFrame(index=lags, columns=INSTR, dtype=float)
for k in lags:
    aligned = ret_w.shift(-k).reindex(sig_w.index)
    for inst in INSTR:
        a = sig_w[inst].astype(float); b = aligned[inst]
        ok = a.notna() & b.notna()
        if ok.sum() > 30 and a[ok].std() > 0 and b[ok].std() > 0:
            lead_lag.loc[k, inst] = a[ok].corr(b[ok])
fig, ax = plt.subplots(figsize=(11, 5))
for inst in INSTR:
    ax.plot(lead_lag.index, lead_lag[inst], marker="o", lw=1.0, label=inst)
ax.axvline(0, color="k", lw=0.5); ax.axhline(0, color="k", lw=0.5)
ax.set_xlabel("lag k:  k>0 => signal leads return (predictive)")
ax.set_ylabel("corr(signal_t, r_{t+k})")
ax.set_title("Lead/lag: is the signal predictive (k>0) or reactive (k<0)?")
ax.legend(ncol=4, fontsize=8); plt.tight_layout(); plt.show()

In [ ]:
# (c) Raw signed-return Sharpe by lag (no geometry / no label filter) — the placebo-in-time check.
from stml.model.evaluate import nav_sharpe
ev = events_frame(matrix)        # [date, instrument, side, sigma] over the released window

def lagged_signed_returns(events, close_wide, lags=(0, 1, 2)):
    out = events[["date", "instrument", "side"]].copy().reset_index(drop=True)
    for L in lags: out[f"g{L}"] = np.nan
    for inst, grp in out.groupby("instrument"):
        if inst not in close_wide.columns: continue
        s = close_wide[inst].dropna(); u = s.pct_change().to_numpy()
        pos = s.index.get_indexer(pd.DatetimeIndex(grp["date"])); side = grp["side"].to_numpy(float)
        for L in lags:
            vals = np.full(len(grp), np.nan)
            ok = (pos >= 0) & (pos + L >= 0) & (pos + L < len(u))
            vals[ok] = u[pos[ok] + L] * side[ok]
            out.loc[grp.index, f"g{L}"] = vals
    return out

def raw_lag_table(G):
    rows = []
    for inst, grp in G.groupby("instrument"):
        for L in (0, 1, 2):
            r = grp[f"g{L}"].to_numpy(float); r = r[np.isfinite(r)]
            sh = nav_sharpe(pd.DataFrame({"ret": r}), np.ones(len(r), bool))["sharpe"]
            rows.append({"instrument": inst, "lag": L, "sharpe": round(sh, 3)})
    return pd.DataFrame(rows).pivot(index="instrument", columns="lag", values="sharpe")

G = lagged_signed_returns(ev, close_w)
print("RAW primary Sharpe by lag (no geometry, no filter) — expect lag 1 strongest (placebo-in-time):")
display(raw_lag_table(G).reindex(INSTR))

### 1.6 — EDA takeaways for the modelling that follows

- **Two windows.** ~30y of prices warm up features; the model lives in the 2020-01→2022-06 signal window.
- **Per-instrument imbalance.** Calibrate / threshold per instrument; watch single-class collapse.
- **Persistence (autocorr 0.6–0.9).** Use **purged + embargoed** blocked CV, never random k-fold.
- **Panel structure (3 correlated blocks).** Treat as a panel; block CV across the whole cross-section.
- **Enter at `t+1`.** The signal predicts the next bar — the labelling entry must respect this.
- **Fat tails.** Prefer robust / tree models and volatility-scaled barriers.

## Section 2 — External macro dataset (feature family F11)

**F11 is the only feature family built from external data.** We sourced a cross-asset macro workbook
(`data/additional_data.xlsx`, Bloomberg-style daily/weekly/monthly series) to give the meta-model
*context* the price-only families cannot see — e.g. whether a counter-trend trade is firing into a calm
tape or a credit-stress spike.

**Hypothesis (deliberately simple).** Rather than data-mine the full workbook, we curated a compact set
**broadly representative of the major cross-asset risk dimensions**, and let the model weight them:

1. equity volatility · 2. interest rates · 3. USD strength · 4. credit spreads ·
5. inflation expectations · 6. energy inventories · 7. manufacturing demand.

In [ ]:
from stml.metamodel.macro_features import (
    load_macro_raw, KEEP, SPREAD_INPUTS, SPREADS, DROPPED, MOMENTUM,
    compute_availability, macro_feature_columns)

# All named series physically present in the workbook (paired date+value columns).
_hdr = pd.read_excel(str(DATA / "additional_data.xlsx"), nrows=1)
all_series = sorted(c for c in _hdr.columns
                    if not str(c).lower().startswith("unnamed") and "date" not in str(c).lower())
# load_macro_raw curates these down to the series F11 actually uses (standalone + spread inputs);
# the dropped series are not loaded.
raw = load_macro_raw(str(DATA / "additional_data.xlsx"))
inv = pd.DataFrame([{"series": k, "n_obs": int(v.notna().sum()),
                     "start": v.dropna().index.min().date(), "end": v.dropna().index.max().date()}
                    for k, v in raw.items()]).sort_values("series").reset_index(drop=True)
print(f"workbook contains {len(all_series)} named macro series; F11 uses {len(raw)} of them "
      f"({len(KEEP)} standalone + {len(SPREAD_INPUTS)} spread inputs), drops {len(DROPPED)}, "
      f"and emits {len(macro_feature_columns())} feature columns")
print("all 22 series:", ", ".join(all_series))
print("\nspans of the series F11 loads:")
display(inv)

### 2.1 — Curation: 22 raw series → 15 effective → 45 columns

From the workbook's 22 named series we **kept 12 standalone**, used **2 only as spread inputs**, built
**3 economically motivated spreads**, and **dropped 8** (redundant with a kept series, or out of scope
for this universe). Each effective series yields **3 columns** — a point-in-time *level* plus two
*momentum* (change) horizons — giving **15 × 3 = 45** F11 columns.

In [ ]:
rows = []
for s, (rcls, desc) in KEEP.items():
    rows.append({"series": s, "role": "standalone", "release": rcls, "captures": desc})
for s in SPREAD_INPUTS:
    rows.append({"series": s, "role": "spread-input only", "release": "", "captures": "used only inside a spread"})
for name, (a, b, rcls, desc) in SPREADS.items():
    rows.append({"series": f"{name} = {a} - {b}", "role": "computed spread", "release": rcls, "captures": desc})
for s in DROPPED:
    rows.append({"series": s, "role": "dropped", "release": "", "captures": "redundant / out-of-scope for this universe"})
curation = pd.DataFrame(rows)
print(f"22 raw series -> {len(KEEP)} standalone + {len(SPREADS)} spreads = {len(KEEP)+len(SPREADS)} effective "
      f"x 3 cols = {len(macro_feature_columns())} F11 columns "
      f"({len(SPREAD_INPUTS)} spread-input-only, {len(DROPPED)} dropped)")
display(curation)

### 2.2 — Point-in-time discipline (no look-ahead)

A macro value can only enter a trade-day row **once it has actually been published**. Each series has a
release cadence that sets both its availability lag and its momentum horizons:

| class | availability rule | momentum (business days) |
|---|---|---|
| `daily` (market series) | same-day EOD close (lag 0) | 5, 20 |
| `weekly_eia` (inventories) | Friday week-ending stamp **+ 6 calendar days** (a conservative buffer past the ~Wed/Thu release) | 5, 20 |
| `monthly_pmi` (PMIs) | month-end stamp **+ 1 business day** (the release) | 21, 63 |

The standardiser (z-score) is **fit on the FE-train slice only (≤ 2021-07-01) and frozen** forward;
the 45 columns are **broadcast identically to all 11 instruments** (global macro context). F11 is
therefore a **fitted (TF)** family under our leakage taxonomy (Section 4).

In [ ]:
# Surface the availability rule straight from the code (one representative stamp per class).
demos = [("daily", pd.Timestamp("2021-08-06")),
         ("weekly_eia", pd.Timestamp("2021-08-06")),   # a Friday week-ending stamp
         ("monthly_pmi", pd.Timestamp("2021-07-31"))]  # a month-end stamp
lag_rows = []
for rcls, stamp in demos:
    avail = compute_availability(stamp, rcls)
    lag_rows.append({"release_class": rcls, "stamp": stamp.date(), "available_on": avail.date(),
                     "lag_days": (avail - stamp).days, "momentum_bdays": MOMENTUM[rcls]})
display(pd.DataFrame(lag_rows))

In [ ]:
# The realised F11 columns in the feature matrix, and proof of the global broadcast.
f11 = [c for c in matrix.columns if c.startswith("f11_")]
by_suffix = collections.Counter(c.rsplit("_", 1)[1] for c in f11)
print(f"{len(f11)} f11_* columns; by suffix: {dict(by_suffix)}")

d0 = matrix["date"].iloc[len(matrix) // 2]
two = (matrix[(matrix.date == d0) & (matrix.instrument.isin(["cl1s", "es1s"]))]
       .set_index("instrument")[f11[:6]].T)
print(f"Same date ({pd.Timestamp(d0).date()}), two different instruments -> identical macro values:")
display(two.round(3))

vix = matrix[matrix.instrument == "cl1s"].set_index("date")["f11_vix_level"].sort_index()
ax = vix.plot(figsize=(11, 2.6), color="C3")
ax.axhline(0, color="k", lw=0.5)
ax.set_title("f11_vix_level (FE-train z-scored) across the released window")
plt.tight_layout(); plt.show()

**How F11 feeds the meta-model.** These 45 standardised, publication-lagged columns join *every*
instrument-date row, so the model can learn macro-conditional adjustments to the per-instrument signal
(e.g. trust a counter-trend trade less when MOVE and HY-OAS are spiking together).

## Section 3 — HMM regime features (families F3 & F17)

Two families summarise the **volatility regime** each row sits in:

- **F3** — a **2-regime** pair: a Gaussian Mixture + a Markov-switching model (high-vol posterior,
  switch intensity, dwell time).
- **F17** — a **3-state Gaussian HMM** (low / mid / high vol) whose **transition matrix** couples the
  states through time.

Both are fit **per instrument** (11 separate models each), on the **FE-train partition only**
(≤ 2021-07-01), and then **frozen** — *not* pooled across the universe (contrast F11) and *not* per
asset class (contrast the F4 latent stack). Both emit strictly **causal, filtered (forward-only)**
posteriors `P(state | data ≤ t)` — never `hmmlearn`'s smoothed `predict_proba`, which would peek at
`t+1…T`. They are **fitted (TF)** families.

**Observation vector (2-D, per instrument):** `ret` = daily log return, and `vol` = trailing 20-day
**annualised** realised volatility. F17 consumes `(ret, vol)` directly; F3's GMM standardises
`(ret, vol)` with frozen FE-train statistics while its Markov model runs on `ret` alone.

In [ ]:
# The one live-fit cell. FeaturePipeline.fit() fits ALL fitted families (F3/F17 per instrument,
# F4 per class, F11 global) on FE-train and freezes them. ~1-3 minutes.
REP_INST = "cl1s"   # representative: energy, full history, strongest signal (label-1 ~ 0.70)
pipe = None
if CAP_HMM:
    from stml.metamodel.pipeline import FeaturePipeline
    pipe = FeaturePipeline(macro_path=str(DATA / "additional_data.xlsx")).fit(ohlcv, signals)
    print(f"FeaturePipeline fitted. Showing {REP_INST} in depth, then an 11-instrument summary.")
else:
    print("hmmlearn unavailable -> skipping the live fit; the materialised f3_*/f17_* columns below still work.")

In [ ]:
# F17 — the 3-state HMM for the representative instrument. ALWAYS reorder arrays by `bundle.order`
# (states sorted by ascending FE-train mean vol) so lo/mid/hi are comparable; raw EM order is arbitrary.
if pipe is not None:
    b = pipe._hmm[REP_INST]; order = b.order; names = ["lo", "mid", "hi"]
    T     = pd.DataFrame(b.hmm.transmat_[np.ix_(order, order)], index=names, columns=names)
    means = pd.DataFrame(b.hmm.means_[order], index=names, columns=["ret", "vol"])
    start = pd.Series(b.hmm.startprob_[order], index=names, name="startprob")
    covs  = np.asarray(b.hmm.covars_)[order]
    disp  = pd.DataFrame({"sd_ret": np.sqrt(covs[:, 0, 0]), "sd_vol": np.sqrt(covs[:, 1, 1])}, index=names)
    print(f"F17 — {REP_INST}: 3-state Gaussian HMM fit on {len(b.train_index)} FE-train days")
    print("\nTransition matrix  P(to | from)   (rows = from-state, cols = to-state):")
    display(T.round(3))
    print("Per-regime emission means  (raw log return, annualised vol) — confirms lo < mid < hi vol:")
    display(means.round(4))
    print("Start probabilities & per-regime dispersion:")
    display(pd.concat([start, disp], axis=1).round(4))
    print("diag(T) =", T.values.diagonal().round(3), "-> high persistence: regimes are sticky.")

In [ ]:
# F3 — the GMM + Markov-switching pair for the same instrument (the two should agree on 'high-vol').
if pipe is not None:
    rb = pipe._regime[REP_INST]
    print(f"F3 — {REP_INST}: high-vol GMM component = {rb.gmm_highvol_comp} | "
          f"high-vol Markov regime = {rb.markov_highvol_regime}")
    gmm_means = pd.DataFrame(rb.gmm.means_, columns=["z_ret", "z_vol"],
                             index=[f"comp{i}" for i in range(rb.gmm.n_components)])
    print("GMM component means (in FROZEN-standardised z-space, not raw units):")
    display(gmm_means.round(3))
    print("Markov-switching variances (last 2 fitted params; the larger one is the high-vol regime):",
          np.round(np.asarray(rb.markov_params[-2:], float), 5))

In [ ]:
# 11-instrument summary: regime persistence (transition-matrix diagonal) + high-state vol per instrument.
if pipe is not None:
    rows = []
    for inst in INSTR:
        b = pipe._hmm[inst]
        rec = {"instrument": inst, "ok": b.ok, "n_train": len(b.train_index)}
        if b.ok:
            d = b.hmm.transmat_[np.ix_(b.order, b.order)].diagonal()
            rec.update({"persist_lo": round(float(d[0]), 3), "persist_mid": round(float(d[1]), 3),
                        "persist_hi": round(float(d[2]), 3),
                        "hi_state_mean_vol": round(float(b.hmm.means_[b.order][-1, 1]), 3)})
        rows.append(rec)
    display(pd.DataFrame(rows).set_index("instrument").reindex(INSTR))

### 3.1 — Derived columns and an example feature vector

Each family contributes **4 columns** to every nonzero-signal row:

- **F17:** `f17_hmm_state_lo`, `f17_hmm_state_mid`, `f17_hmm_state_hi` (filtered posteriors — a simplex
  that sums to 1) and `f17_hmm_state_argmax` (the most-likely state, nominal 0/1/2).
- **F3:** `f3_gmm_prob_highvol`, `f3_markov_prob_highvol`, `f3_markov_switch_prob` (the trailing |Δ| of
  the high-vol probability — switch intensity) and `f3_regime_dwell` (days since the regime call last
  flipped).

These read straight out of the materialised matrix (no fit needed):

In [ ]:
hmm_cols = ["f3_gmm_prob_highvol", "f3_markov_prob_highvol", "f3_markov_switch_prob", "f3_regime_dwell",
            "f17_hmm_state_lo", "f17_hmm_state_mid", "f17_hmm_state_hi", "f17_hmm_state_argmax"]
row = matrix[matrix.instrument == REP_INST].dropna(subset=["f17_hmm_state_lo"]).iloc[100]
ex = row[["date", "instrument"] + hmm_cols].to_frame("value")
display(ex)
simplex = float(row[["f17_hmm_state_lo", "f17_hmm_state_mid", "f17_hmm_state_hi"]].sum())
print(f"F17 posteriors sum to {simplex:.6f} (simplex); argmax state = {int(row['f17_hmm_state_argmax'])} "
      f"(0=lo, 1=mid, 2=hi vol)")

**How they feed the meta-model.** These 8 causal, frozen columns join every row so the model can learn
that the primary signal's reliability is **regime-dependent** (e.g. counter-trend tends to work in calm
regimes and break during high-vol transitions). Whether they survive cluster-level importance is tested
downstream in `metamodel.ipynb` §7.

## Section 4 — Feature engineering (families F1–F17)

The full feature layer is **175 features + 4 metadata = 179 columns** over **4,984 nonzero-signal
trade-days**, organised into **16 families** (`F1`–`F17`, with `F14` intentionally unused). It is a
de-duplicated **union of three teammates' branches** — the `F1`–`F11` base plus folded-in novel
families (`F12`/`F17` and a Rogers–Satchell vol estimator; `F13`/`F15`/`F16` and several `F5`/`F7`/`F9`
additions).

In [ ]:
from stml.metamodel.catalog import CATALOG, _FAMILY_TITLES
META = {"date", "instrument", "partition", "fe_train_end_date"}
feat = [c for c in matrix.columns if c not in META]

def family_of(col):
    name = col[2:] if col.startswith("z_") else col      # strip a z-twin prefix
    return re.match(r"(f\d+)_", name).group(1).upper()

recs = {}
for c in feat:
    f = family_of(c); lk = CATALOG[c].leakage_class
    r = recs.setdefault(f, {"n": 0, "E": 0, "TF": 0, "LI": 0})
    r["n"] += 1; r[lk] += 1
order = sorted(recs, key=lambda f: int(f[1:]))
fam_tbl = pd.DataFrame([{
    "family": f, "title": _FAMILY_TITLES[f].split(" — ", 1)[1], "n_cols": recs[f]["n"],
    "leakage": "/".join(f"{k}:{recs[f][k]}" for k in ("E", "TF", "LI") if recs[f][k]),
} for f in order]).set_index("family")
display(fam_tbl)

tot = collections.Counter(CATALOG[c].leakage_class for c in feat)
print(f"{len(feat)} features = E:{tot['E']} engineered + TF:{tot['TF']} fitted + LI:{tot['LI']} label-interface")

### 4.1 — Leakage taxonomy (the methodology backbone)

Every feature uses only information available at or before time `t`. Each column is classified into one
of three leakage classes (proven in the test suite):

- **E — engineered.** No fitting; causal by **truncation-invariance** (the value at `t` is identical
  whether computed on `data[:t+1]` or the full series).
- **TF — fitted.** GMM / Markov / HMM / PCA / KMeans / autoencoder / scaler families (`F3`, `F4`,
  `F11`, `F16`, `F17`) are fit on the **FE-train partition only (≤ 2021-07-01)** and applied causally
  with **frozen** parameters.
- **LI — label-interface.** The two columns the downstream triple-barrier label consumes:
  `f2_vol_20` (the barrier sigma) and `f5_trailing_run_length`.

Additionally, every scale-dependent **E** column carries a parallel **`z_<col>` twin**: a per-instrument
*causal expanding-window* z-score (split-agnostic). Bounded columns (ratios, probabilities, sin/cos,
percentiles) get no twin.

### 4.2 — The families, grouped (and tied back to the EDA)

- **Signal-aligned core (highest value).** `F1` counter-trend / mean-reversion (led by `f1_mr_score_20`
  — the family the §1.5 lag-1 evidence predicts should dominate), `F5` signal-derived run structure
  (motivated by the §1.4 persistence), `F6` momentum / trend-contrast.
- **Volatility & path.** `F2` volatility & dispersion (incl. the Rogers–Satchell estimator and z-twins),
  `F12` Hurst / variance-ratio / efficiency-ratio path structure, `F13` wavelet multiscale energy,
  `F15` conditional-risk / first-passage.
- **Regime & drift (fitted).** `F3` GMM+Markov and `F17` HMM regimes (Section 3), `F16` concept-drift
  alignment, `F4` latent PCA / KMeans / autoencoder (fit per asset class).
- **Structure & context.** `F7` microstructure (volume / open-interest), `F10` OHLC price-action,
  `F8` calendar sin/cos, `F9` cross-sectional rank / pair-correlation (motivated by the §1.3 blocks and
  the low cross-signal correlation), `F11` macro context (Section 2).

(`F14` is intentionally skipped — a gap left when the three branches were merged.) Full per-column
documentation lives in [`reports/feature-catalog.md`](../../reports/feature-catalog.md).

### 4.3 — How the matrix is built

```python
from stml.io import load_clean_data
from stml.metamodel import FeaturePipeline
ohlcv, signals = load_clean_data()
matrix = FeaturePipeline().fit(ohlcv, signals).transform(ohlcv, signals)
```

`.fit()` fixes the chronological train/val/test split, fits the **TF** families on the FE-train block
only, and freezes every learned parameter. `.transform()` applies the engineered **E** families and the
frozen **TF** transforms causally, adds the cross-sectional and z-twin columns, **restricts rows to
nonzero-signal trade-days**, tags each row's `partition` and `fe_train_end_date`, and preserves
structural NaNs (a closed venue is information — never forward-filled). The CLI entry point is
`python -m stml.metamodel.build_features`; the artifact is `results/feature_matrix.parquet`.

In [ ]:
# Provenance check — the materialised artifact matches the pipeline contract exactly.
print("provenance:")
for k in ["fe_train_end_date", "seed", "n_rows", "n_feature_cols"]:
    print(f"  {k}: {prov[k]}")
print(f"  partition_row_counts: {prov['partition_row_counts']}")
print(f"  macro.n_macro_features: {prov['macro']['n_macro_features']}")
assert matrix.shape == (prov["n_rows"], prov["n_feature_cols"] + 4)
print(f"OK — matrix {matrix.shape} == {prov['n_rows']} rows x ({prov['n_feature_cols']} features + 4 metadata)")

---

**Next (later parts of this notebook).** With the foundations in place — clean data, the macro context,
the regime features, and the full feature catalogue — the remaining sections cover **triple-barrier
labelling**, **meta-model fitting & comparison** (linear / tree / neural families under purged CV), and
**position-sizing / weight extraction** from the predicted probabilities.